**Arthur L'AFFÉTER, Landry LHOMME, Tiago DA COSTA, Tom LEPRIEUR**

---

# Classificateur d'Émotions Félines — Projet Data

**Problématique :** Développer un classificateur d'émotions félines à partir d'images pour aider propriétaires et vétérinaires à détecter le bien-être et le stress des chats lors des interactions humaines.

**Dataset :** Cat Emotions Dataset (Roboflow) — 671 images, 7 classes
**Auteurs :** *(Noms et prénoms des membres du groupe)*
**Date :** Mars 2026

---

## Table des matières

1. [Introduction & Contexte métier](#1)
2. [Collecte et compréhension des données](#2)
3. [Nettoyage et prétraitement](#3)
4. [Analyse exploratoire (EDA)](#4)
5. [Baseline — Random Forest + HOG](#5)
6. [Modèle 1 — CNN from scratch](#6)
7. [Modèle 2 — Transfer Learning MobileNetV2](#7)
8. [Comparaison des performances](#8)
9. [Explicabilité — Grad-CAM](#9)
10. [Discussion : limites, biais, améliorations](#10)
11. [Conclusion et recommandations](#11)


In [ ]:
# ================================================================
# IMPORTS ET CONFIGURATION GLOBALE
# ================================================================
import os
import sys
import random
import warnings
import json
from pathlib import Path
from collections import Counter

# Données & calcul
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns
from PIL import Image

# Machine Learning classique
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder

# Features images
from skimage.feature import hog
from skimage.color import rgb2gray

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mnv2_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

warnings.filterwarnings('ignore')

# ── Seeds pour reproductibilité ──────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── Constantes globales ──────────────────────────────────────
IMG_SIZE    = 128          # Résolution cible : 128×128 px
BATCH_SIZE  = 32
EPOCHS_CNN  = 40
EPOCHS_P1   = 15           # MobileNetV2 phase 1 (têtes figées)
EPOCHS_P2   = 20           # MobileNetV2 phase 2 (fine-tuning)
NUM_CLASSES = 7
CLASSES     = ['Angry', 'Disgusted', 'Happy', 'Normal', 'Sad', 'Scared', 'Surprised']

# ── Chemins ──────────────────────────────────────────────────
BASE_DIR  = Path('.')
TRAIN_DIR = BASE_DIR / 'train'
VALID_DIR = BASE_DIR / 'valid'

print(f"TensorFlow : {tf.__version__}")
print(f"Python     : {sys.version.split()[0]}")
print(f"GPU dispo  : {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Classes    : {CLASSES}")


---
<a id='1'></a>
## 1. Introduction & Contexte métier

### Problématique

Comprendre les émotions des chats est un défi pour leurs propriétaires et les professionnels vétérinaires. Un chat stressé ou douloureux peut adopter des postures et des expressions faciales spécifiques (*oreilles aplaties*, *yeux dilatés*, *moustaches tendues*) qui, mal interprétées, conduisent à des erreurs de prise en charge.

**Objectif :** construire un système de classification automatique d'images permettant de reconnaître 7 états émotionnels félins (*Angry, Disgusted, Happy, Normal, Sad, Scared, Surprised*) avec une précision exploitable en situation réelle.

### Valeur métier

| Acteur | Bénéfice attendu |
|---|---|
| Propriétaire | Détecter précocement le stress/malaise de l'animal |
| Vétérinaire | Aide au diagnostic comportemental lors des consultations |
| Refuge / shelter | Suivi automatisé du bien-être en cage |
| Industrie PetTech | Intégration dans caméras connectées (gamification) |

### Hypothèses de départ

1. Les expressions faciales (position des oreilles, ouverture des yeux, tension des moustaches) constituent des signaux visuels discriminants entre les classes.
2. 671 images sont insuffisantes pour entraîner un CNN from scratch de haute performance → la data augmentation et le transfer learning seront essentiels.
3. Certaines émotions visuellement proches (*Scared* vs *Angry*, *Normal* vs *Relaxed*) généreront des confusions — la matrice de confusion le révélera.

### Variable cible

`emotion` — variable catégorielle à 7 modalités (classification multi-classes).


---
<a id='2'></a>
## 2. Collecte et compréhension des données

Le dataset est issu de **Roboflow Universe** (Cat Emotions Dataset), pré-divisé en dossiers `train/` et `valid/`, chacun contenant un sous-dossier par classe.

- **Format :** JPEG, noms de fichiers horodatés avec hash Roboflow
- **Résolution :** variable (majoritairement 640×640)
- **Pas de test set fourni** → nous effectuerons un re-split stratifié (70/15/15)


In [ ]:
# ── Fonction utilitaire : scan d'un split ───────────────────
def scan_split(root_dir):
    stats = {}
    for cls_dir in sorted(Path(root_dir).iterdir()):
        if cls_dir.is_dir():
            imgs = [f for f in cls_dir.iterdir()
                    if f.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp')]
            stats[cls_dir.name] = imgs
    return stats

train_raw = scan_split(TRAIN_DIR)
valid_raw = scan_split(VALID_DIR)

# ── Résumé tabulaire ─────────────────────────────────────────
rows = []
for cls in CLASSES:
    t = len(train_raw.get(cls, []))
    v = len(valid_raw.get(cls, []))
    rows.append({'Classe': cls, 'Train (brut)': t, 'Valid (brut)': v, 'Total': t + v})

df_counts = pd.DataFrame(rows)
print(df_counts.to_string(index=False))
print(f"\nTotal images : {df_counts['Total'].sum()}")
print(f"Répertoire   : {BASE_DIR.resolve()}")


In [ ]:
# ── Inspection d'une image type ─────────────────────────────
sample_path = list(train_raw['Happy'])[0]
with Image.open(sample_path) as img:
    print(f"Mode          : {img.mode}")
    print(f"Taille        : {img.size}  (W×H)")
    print(f"Format        : {img.format}")
    arr = np.array(img)
    print(f"Array shape   : {arr.shape}")
    print(f"Valeurs min/max/mean : {arr.min()} / {arr.max()} / {arr.mean():.1f}")


---
<a id='3'></a>
## 3. Nettoyage et prétraitement

### Stratégie

| Étape | Décision | Justification |
|---|---|---|
| Re-split | 70 / 15 / 15 (stratifié) | Assure la représentativité de chaque classe dans chaque split |
| Résolution | 128×128 px | Compromis qualité/vitesse adapté à un dataset de 671 images |
| Normalisation | `/255.` (CNN scratch) ou `preprocess_input` MobileNetV2 | Chaque backbone a ses propres attentes |
| Augmentation | Flip H, rotation ±15°, zoom ±10%, brightness ±20% | Combat le surapprentissage sur petit dataset |
| Valeurs manquantes | Détection fichiers corrompus | Garantit la reproductibilité |


In [ ]:
# ── Collecte de tous les chemins + labels ────────────────────
all_paths, all_labels = [], []

for split_dir in [TRAIN_DIR, VALID_DIR]:
    for cls_dir in sorted(Path(split_dir).iterdir()):
        if cls_dir.is_dir() and cls_dir.name in CLASSES:
            for img_path in cls_dir.iterdir():
                if img_path.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp'):
                    all_paths.append(str(img_path))
                    all_labels.append(cls_dir.name)

print(f"Total fichiers collectés : {len(all_paths)}")
print(f"Distribution : {Counter(all_labels)}")


In [ ]:
# ── Détection des images corrompues ─────────────────────────
corrupted = []
for p in all_paths:
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception as e:
        corrupted.append((p, str(e)))

if corrupted:
    print(f"⚠ {len(corrupted)} images corrompues détectées — retirées.")
    bad_set = {c[0] for c in corrupted}
    all_paths = [p for p in all_paths if p not in bad_set]
    all_labels = [l for p, l in zip(all_paths + list(bad_set), all_labels + ['?'] * len(bad_set))
                  if p not in bad_set]
else:
    print(f"✓ Aucune image corrompue ({len(all_paths)} images valides).")


In [ ]:
# ── Re-split stratifié 70 / 15 / 15 ────────────────────────
# Étape 1 : 70% train  |  30% temp
paths_train, paths_temp, labels_train, labels_temp = train_test_split(
    all_paths, all_labels,
    test_size=0.30, random_state=SEED, stratify=all_labels
)

# Étape 2 : 15% valid  |  15% test  (50/50 du temp)
paths_val, paths_test, labels_val, labels_test = train_test_split(
    paths_temp, labels_temp,
    test_size=0.50, random_state=SEED, stratify=labels_temp
)

print(f"Train : {len(paths_train)} images")
print(f"Valid : {len(paths_val)} images")
print(f"Test  : {len(paths_test)} images")
print(f"Total : {len(paths_train) + len(paths_val) + len(paths_test)}")

# Vérification de la distribution par split
for split_name, split_labels in [('Train', labels_train), ('Valid', labels_val), ('Test', labels_test)]:
    c = Counter(split_labels)
    row = ' | '.join(f"{cls[:3]}:{c[cls]}" for cls in CLASSES)
    print(f"{split_name}: {row}")


In [ ]:
# ── Fonction de chargement d'images ─────────────────────────
def load_images(paths, labels, img_size=IMG_SIZE, preprocess_fn=None):
    """Charge et redimensionne une liste d'images.

    Args:
        paths       : liste de chemins fichiers
        labels      : liste de labels correspondants
        img_size    : taille cible (carré)
        preprocess_fn : fonction de preprocessing (None = normalise /255)
    Returns:
        X (np.array float32), y (np.array int)
    """
    le = LabelEncoder()
    le.fit(CLASSES)

    X, y = [], []
    for path, label in zip(paths, labels):
        try:
            img = load_img(path, target_size=(img_size, img_size))
            arr = img_to_array(img)  # float32 [0,255]
            if preprocess_fn is not None:
                arr = preprocess_fn(arr)
            else:
                arr = arr / 255.0
            X.append(arr)
            y.append(label)
        except Exception:
            pass

    X = np.array(X, dtype=np.float32)
    y = le.transform(y)
    return X, y

# Encodeur de labels (réutilisé globalement)
label_encoder = LabelEncoder()
label_encoder.fit(CLASSES)

print("Fonctions de chargement définies.")
print(f"Mapping classes : {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")


In [ ]:
# ── ImageDataGenerator pour CNN scratch ──────────────────────
# Choix d'augmentation :
#   - flip horizontal : réaliste (chat peut être vu de gauche ou droite)
#   - rotation ±15°   : variation naturelle de prise de vue
#   - zoom ±10%       : simule distance caméra
#   - brightness       : variation d'éclairage (flash vétérinaire, lumière naturelle)
# Pas de flip vertical car un chat la tête en bas est rare et nuirait aux features

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.10,
    brightness_range=[0.80, 1.20],
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)   # Pas d'augmentation en validation/test

# Générateurs à partir des dossiers d'origine
# On utilise un dossier temporaire structuré pour flow_from_directory
# Plus simple : on charge en mémoire vu la taille du dataset (671 images)

print("ImageDataGenerators définis.")
print("Augmentation train : flip H, rotation 15°, zoom 10%, brightness ±20%")


---
<a id='4'></a>
## 4. Analyse Exploratoire des Données (EDA)

L'EDA poursuit quatre objectifs :
1. **Quantifier le déséquilibre** de classes → impactera le choix des métriques et le traitement du déséquilibre
2. **Visualiser des exemples** par émotion → valider la qualité des annotations
3. **Analyser les statistiques de pixels** → guider la normalisation
4. **Identifier les biais potentiels** (fond, éclairage, couleur du pelage)


In [ ]:
# ── 4.1 Distribution des classes ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = Counter(all_labels)
counts = [class_counts[c] for c in CLASSES]
colors = sns.color_palette('husl', NUM_CLASSES)

# Barplot
axes[0].bar(CLASSES, counts, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribution des classes — dataset complet', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Émotion')
axes[0].set_ylabel("Nombre d'images")
axes[0].tick_params(axis='x', rotation=30)
for i, (cls, cnt) in enumerate(zip(CLASSES, counts)):
    axes[0].text(i, cnt + 1, str(cnt), ha='center', fontsize=9)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts, labels=CLASSES, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}
)
axes[1].set_title('Répartition en pourcentage', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

# Indice de déséquilibre (ratio max/min)
ratio = max(counts) / min(counts)
print(f"Classe la plus représentée : {CLASSES[np.argmax(counts)]} ({max(counts)} imgs)")
print(f"Classe la moins représentée : {CLASSES[np.argmin(counts)]} ({min(counts)} imgs)")
print(f"Ratio déséquilibre max/min : {ratio:.2f}x")
print("→ Déséquilibre modéré — F1-macro sera notre métrique principale (robuste au déséquilibre)")


In [ ]:
# ── 4.2 Exemples visuels par classe ─────────────────────────
# Affiche 5 exemples par émotion pour valider la qualité des annotations
N_EXAMPLES = 5
fig, axes = plt.subplots(NUM_CLASSES, N_EXAMPLES, figsize=(14, 22))
fig.suptitle('Exemples visuels par émotion (5 images par classe)',
             fontsize=15, fontweight='bold', y=1.01)

for row_i, cls in enumerate(CLASSES):
    cls_paths = [p for p, l in zip(all_paths, all_labels) if l == cls]
    sample_paths = random.sample(cls_paths, min(N_EXAMPLES, len(cls_paths)))

    for col_i, img_path in enumerate(sample_paths):
        ax = axes[row_i][col_i]
        img = Image.open(img_path).resize((128, 128))
        ax.imshow(img)
        ax.axis('off')
        if col_i == 0:
            ax.set_ylabel(cls, fontsize=11, fontweight='bold', rotation=0,
                          labelpad=60, va='center')

plt.tight_layout()
plt.savefig('eda_samples_per_class.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.3 Analyse des résolutions et tailles ──────────────────
widths, heights = [], []
for p in all_paths[:200]:   # Sous-échantillon pour la rapidité
    try:
        with Image.open(p) as im:
            w, h = im.size
            widths.append(w)
            heights.append(h)
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution largeurs (px)', fontweight='bold')
axes[0].set_xlabel('Largeur (px)')

axes[1].hist(heights, bins=20, color='coral', edgecolor='white')
axes[1].set_title('Distribution hauteurs (px)', fontweight='bold')
axes[1].set_xlabel('Hauteur (px)')

axes[2].scatter(widths, heights, alpha=0.4, color='purple', s=20)
axes[2].set_title('Largeur vs Hauteur', fontweight='bold')
axes[2].set_xlabel('Largeur (px)')
axes[2].set_ylabel('Hauteur (px)')

plt.tight_layout()
plt.savefig('eda_resolutions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Largeur  : min={min(widths)}px  max={max(widths)}px  médiane={int(np.median(widths))}px")
print(f"Hauteur  : min={min(heights)}px  max={max(heights)}px  médiane={int(np.median(heights))}px")
print("→ Images de résolutions variées → redimensionnement obligatoire à 128×128")


In [ ]:
# ── 4.4 Analyse des intensités de pixels par classe ─────────
# On compare la luminosité moyenne par classe pour détecter des biais d'éclairage

fig, ax = plt.subplots(figsize=(13, 5))
palette = sns.color_palette('husl', NUM_CLASSES)

for i, cls in enumerate(CLASSES):
    cls_paths = [p for p, l in zip(all_paths, all_labels) if l == cls]
    means = []
    for p in cls_paths[:40]:   # 40 images par classe suffisent
        try:
            arr = np.array(Image.open(p).resize((64, 64)).convert('RGB'))
            means.append(arr.mean())
        except Exception:
            pass
    ax.violinplot(means, positions=[i], showmedians=True,
                  widths=0.6)
    # Coloration manuelle
    for pc in ax.collections[-2:]:
        pc.set_facecolor(palette[i])
        pc.set_alpha(0.6)

ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=20)
ax.set_title('Distribution de la luminosité moyenne par classe', fontsize=13, fontweight='bold')
ax.set_ylabel('Luminosité moyenne (0–255)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('eda_pixel_intensity.png', dpi=120, bbox_inches='tight')
plt.show()
print("→ Des différences de luminosité existent → normalisation /255 indispensable")


In [ ]:
# ── 4.5 Synthèse EDA ────────────────────────────────────────
print("=" * 55)
print("SYNTHÈSE EDA")
print("=" * 55)
print(f"  • {len(all_paths)} images valides, 7 classes")
print(f"  • Déséquilibre modéré ({ratio:.1f}x) → métrique F1-macro")
print("  • Résolutions variables → resize 128×128 obligatoire")
print("  • Luminosité variable entre classes → normalisation requise")
print("  • Dataset petit (671 imgs) → data augmentation essentielle")
print("  • Annotations visuellement cohérentes (vérifiées manuellement)")
print("  • Biais potentiel : fond uni vs complexe, couleur du pelage")
print("=" * 55)


---
<a id='5'></a>
## 5. Baseline — Random Forest + HOG

### Justification du choix

Avant d'entraîner des réseaux de neurones coûteux, on établit une **baseline non-neuronale** :

- **HOG (Histogram of Oriented Gradients)** : descripteur classique qui capture les contours et gradients locaux — pertinent pour les expressions faciales (contours des oreilles, des yeux).
- **Random Forest** : robuste aux petits datasets, peu d'hyperparamètres critiques, résultats interprétables (feature importances).

Cette baseline fixe un plancher de performance que les CNN devront dépasser pour justifier leur complexité.


In [ ]:
# ── Chargement des images en mémoire (niveaux de gris) ───────
print("Extraction des features HOG en cours...")

def extract_hog_features(paths, labels, img_size=64):
    """Extrait les features HOG pour une liste d'images.
    On réduit à 64×64 pour HOG (taille standard HOG).
    """
    features, y = [], []
    for path, label in zip(paths, labels):
        try:
            img = load_img(path, target_size=(img_size, img_size), color_mode='grayscale')
            arr = img_to_array(img).squeeze() / 255.0
            feat = hog(arr,
                       orientations=9,
                       pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2),
                       block_norm='L2-Hys',
                       feature_vector=True)
            features.append(feat)
            y.append(label)
        except Exception:
            pass
    return np.array(features), np.array(y)

X_train_hog, y_train_hog = extract_hog_features(paths_train, labels_train)
X_val_hog,   y_val_hog   = extract_hog_features(paths_val,   labels_val)
X_test_hog,  y_test_hog  = extract_hog_features(paths_test,  labels_test)

# Encodage labels
y_train_hog_enc = label_encoder.transform(y_train_hog)
y_val_hog_enc   = label_encoder.transform(y_val_hog)
y_test_hog_enc  = label_encoder.transform(y_test_hog)

print(f"Shape features HOG : {X_train_hog.shape}  ({X_train_hog.shape[1]} features par image)")
print(f"Train : {len(X_train_hog)} | Val : {len(X_val_hog)} | Test : {len(X_test_hog)}")


In [ ]:
# ── Entraînement Random Forest ───────────────────────────────
# GridSearch léger sur les hyperparamètres clés
param_grid = {
    'n_estimators': [100, 200],
    'max_depth':    [None, 20],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(random_state=SEED, class_weight='balanced', n_jobs=-1)

gs_rf = GridSearchCV(
    rf_base, param_grid,
    cv=3, scoring='f1_macro',
    n_jobs=-1, verbose=0
)
gs_rf.fit(X_train_hog, y_train_hog_enc)

rf_best = gs_rf.best_estimator_
print(f"Meilleurs hyperparamètres RF : {gs_rf.best_params_}")
print(f"F1-macro CV (train)           : {gs_rf.best_score_:.4f}")


In [ ]:
# ── Évaluation Random Forest sur test ───────────────────────
y_pred_rf = rf_best.predict(X_test_hog)

acc_rf = accuracy_score(y_test_hog_enc, y_pred_rf)
f1_rf  = f1_score(y_test_hog_enc, y_pred_rf, average='macro')

print(f"Random Forest — Accuracy : {acc_rf:.4f}  |  F1-macro : {f1_rf:.4f}")
print()
print(classification_report(y_test_hog_enc, y_pred_rf,
                              target_names=CLASSES, digits=3))

# Matrice de confusion RF
fig, ax = plt.subplots(figsize=(9, 7))
cm_rf = confusion_matrix(y_test_hog_enc, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title(f'Matrice de confusion — Random Forest (F1={f1_rf:.3f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


---
<a id='6'></a>
## 6. Modèle 1 — CNN from scratch (Keras Sequential)

### Architecture choisie

| Bloc | Couches | Justification |
|---|---|---|
| Entrée | 128×128×3 | Bon compromis résolution/mémoire |
| Conv Bloc 1 | Conv2D(32) + BN + ReLU + MaxPool | Capture les features bas niveau (bords, textures) |
| Conv Bloc 2 | Conv2D(64) + BN + ReLU + MaxPool | Features intermédiaires (formes) |
| Conv Bloc 3 | Conv2D(128) + BN + ReLU + MaxPool | Features haut niveau (structures faciales) |
| Conv Bloc 4 | Conv2D(256) + BN + ReLU + GAP | Réduction spatiale sans perte d'information |
| Tête | Dense(256) + Dropout(0.5) + Softmax(7) | Classification finale |

**BatchNormalization** : stabilise l'entraînement et agit comme régularisateur léger.
**GlobalAveragePooling** : réduit le surapprentissage vs Flatten sur petit dataset.
**Dropout(0.5)** : régularisation principale sur un dataset de 671 images.


In [ ]:
# ── Chargement des images en mémoire (normalisées /255) ───────
print("Chargement des images pour CNN scratch...")

X_train_cnn, y_train_cnn = load_images(paths_train, labels_train)
X_val_cnn,   y_val_cnn   = load_images(paths_val,   labels_val)
X_test_cnn,  y_test_cnn  = load_images(paths_test,  labels_test)

print(f"X_train : {X_train_cnn.shape}  y_train : {y_train_cnn.shape}")
print(f"X_val   : {X_val_cnn.shape}  y_val   : {y_val_cnn.shape}")
print(f"X_test  : {X_test_cnn.shape}  y_test  : {y_test_cnn.shape}")


In [ ]:
# ── Architecture CNN from scratch ───────────────────────────
def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """CNN 4 blocs convolutifs avec BatchNorm et GlobalAveragePooling."""
    model = models.Sequential([
        # ─── Augmentation embarquée (active seulement à l'entraînement) ───
        layers.RandomFlip('horizontal', input_shape=input_shape),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.08),
        layers.RandomBrightness(0.15),

        # ─── Bloc 1 : 32 filtres ─────────────────────────────────────────
        layers.Conv2D(32, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),          # 64×64

        # ─── Bloc 2 : 64 filtres ─────────────────────────────────────────
        layers.Conv2D(64, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),          # 32×32

        # ─── Bloc 3 : 128 filtres ────────────────────────────────────────
        layers.Conv2D(128, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D(2),          # 16×16

        # ─── Bloc 4 : 256 filtres ────────────────────────────────────────
        layers.Conv2D(256, 3, padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.GlobalAveragePooling2D(), # → (256,)

        # ─── Tête de classification ───────────────────────────────────────
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.50),
        layers.Dense(num_classes, activation='softmax')
    ], name='CatEmotions_CNN')

    return model

cnn_model = build_cnn()
cnn_model.summary()


In [ ]:
# ── Compilation + callbacks ──────────────────────────────────
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_cnn_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=0
    )
]

print("Modèle compilé — prêt à l'entraînement")


In [ ]:
# ── Entraînement CNN ─────────────────────────────────────────
# On utilise les données chargées en mémoire avec augmentation via ImageDataGenerator
# Avantage : contrôle total du pipeline, pas besoin de dossiers temporaires

train_gen = train_datagen.flow(X_train_cnn, y_train_cnn,
                               batch_size=BATCH_SIZE, seed=SEED)
val_gen   = val_datagen.flow(X_val_cnn, y_val_cnn,
                             batch_size=BATCH_SIZE, shuffle=False)

print(f"Entraînement CNN ({EPOCHS_CNN} époques max, early stopping patience=10)...")

history_cnn = cnn_model.fit(
    train_gen,
    epochs=EPOCHS_CNN,
    validation_data=val_gen,
    callbacks=cnn_callbacks,
    verbose=1
)

print("\n✓ Entraînement terminé.")


In [ ]:
# ── Courbes d'apprentissage CNN ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hist = history_cnn.history
epochs_ran = range(1, len(hist['loss']) + 1)

# Loss
axes[0].plot(epochs_ran, hist['loss'],     label='Train loss',     color='royalblue')
axes[0].plot(epochs_ran, hist['val_loss'], label='Val loss',       color='tomato',  linestyle='--')
axes[0].set_title('Loss — CNN from scratch', fontweight='bold')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Loss (cross-entropie)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, hist['accuracy'],     label='Train accuracy', color='royalblue')
axes[1].plot(epochs_ran, hist['val_accuracy'], label='Val accuracy',   color='tomato',  linestyle='--')
axes[1].set_title('Accuracy — CNN from scratch', fontweight='bold')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cnn_learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Évaluation CNN sur test ──────────────────────────────────
y_pred_cnn_proba = cnn_model.predict(X_test_cnn, verbose=0)
y_pred_cnn = np.argmax(y_pred_cnn_proba, axis=1)

acc_cnn = accuracy_score(y_test_cnn, y_pred_cnn)
f1_cnn  = f1_score(y_test_cnn, y_pred_cnn, average='macro')

print(f"CNN from scratch — Accuracy : {acc_cnn:.4f}  |  F1-macro : {f1_cnn:.4f}")
print()
print(classification_report(y_test_cnn, y_pred_cnn,
                              target_names=CLASSES, digits=3))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(9, 7))
cm_cnn = confusion_matrix(y_test_cnn, y_pred_cnn)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_cnn, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title(f'Matrice de confusion — CNN scratch (F1={f1_cnn:.3f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


---
<a id='7'></a>
## 7. Modèle 2 — Transfer Learning MobileNetV2

### Justification du choix de MobileNetV2

| Critère | MobileNetV2 | Alternative (VGG16/ResNet50) |
|---|---|---|
| Paramètres | ~3.4M | 138M / 25M |
| Vitesse inférence | Très rapide | Lente / Moyenne |
| Précision ImageNet | 72% Top-1 | 71% / 76% |
| Adapté au dataset petit | ✓ Excellent (léger, peu overfit) | ✗ Risque overfit |

**Stratégie deux phases :**
1. **Phase 1 — Feature Extraction** : on gèle toutes les couches MobileNetV2 et on entraîne seulement la nouvelle tête de classification → apprentissage rapide sans dégrader les poids ImageNet.
2. **Phase 2 — Fine-tuning** : on dégèle les 30 dernières couches et on reprend l'entraînement avec un LR très faible (1e-5) → adaptation fine des features aux expressions félines.


In [ ]:
# ── Chargement images pré-traitées pour MobileNetV2 ─────────
# MobileNetV2 attend des valeurs dans [-1, +1] via preprocess_input
print("Chargement images pour MobileNetV2...")

X_train_mnv2, y_train_mnv2 = load_images(paths_train, labels_train,
                                           preprocess_fn=mnv2_preprocess)
X_val_mnv2,   y_val_mnv2   = load_images(paths_val,   labels_val,
                                           preprocess_fn=mnv2_preprocess)
X_test_mnv2,  y_test_mnv2  = load_images(paths_test,  labels_test,
                                           preprocess_fn=mnv2_preprocess)

print(f"X_train : {X_train_mnv2.shape}  range : [{X_train_mnv2.min():.2f}, {X_train_mnv2.max():.2f}]")
print("→ Valeurs dans [-1, +1] ✓ (attendues par MobileNetV2)")


In [ ]:
# ── Construction du modèle MobileNetV2 ──────────────────────
def build_mobilenetv2(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    """Transfer Learning avec MobileNetV2.
    Couches de base gelées (phase 1), tête de classification personnalisée.
    """
    # Backbone pré-entraîné sur ImageNet, sans la tête de classification
    base = MobileNetV2(
        input_shape=input_shape,
        include_top=False,       # On retire la couche Dense finale ImageNet
        weights='imagenet',
        pooling=None
    )
    base.trainable = False       # Phase 1 : backbone gelé

    # Construction du modèle complet
    inputs = keras.Input(shape=input_shape)

    # Augmentation embarquée (active seulement en training)
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.08)(x)
    x = layers.RandomZoom(0.08)(x)

    x = base(x, training=False)  # training=False : BN du backbone reste en inférence
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='CatEmotions_MobileNetV2')
    return model, base

mnv2_model, mnv2_base = build_mobilenetv2()
mnv2_model.summary()
print(f"\nParamètres entraînables (phase 1) : {mnv2_model.count_params():,}")
print(f"Paramètres gelés (backbone)        : {sum(~w.trainable for w in mnv2_model.weights):,} poids")


In [ ]:
# ── Phase 1 : Feature Extraction ────────────────────────────
mnv2_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_p1 = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=4, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint('best_mnv2_model.keras', monitor='val_accuracy',
                               save_best_only=True, verbose=0)
]

# Générateurs avec augmentation pour MobileNetV2
# Note : les images sont déjà en [-1,+1], on n'applique PAS rescale
aug_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=12,
    zoom_range=0.08,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

train_gen_mnv2 = aug_datagen.flow(X_train_mnv2, y_train_mnv2,
                                   batch_size=BATCH_SIZE, seed=SEED)
val_gen_mnv2   = ImageDataGenerator().flow(X_val_mnv2, y_val_mnv2,
                                            batch_size=BATCH_SIZE, shuffle=False)

print(f"Phase 1 — Feature extraction ({EPOCHS_P1} époques max)...")
history_p1 = mnv2_model.fit(
    train_gen_mnv2,
    epochs=EPOCHS_P1,
    validation_data=val_gen_mnv2,
    callbacks=cb_p1,
    verbose=1
)
print("\n✓ Phase 1 terminée.")


In [ ]:
# ── Phase 2 : Fine-tuning ─────────────────────────────────
# On dégèle les 30 dernières couches du backbone (≈ blocs 14-16 de MobileNetV2)
# Ces blocs contiennent des features de haut niveau potentiellement généralisables

mnv2_base.trainable = True
fine_tune_at = len(mnv2_base.layers) - 30

for layer in mnv2_base.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for w in mnv2_model.trainable_weights)
print(f"Couches entraînables (phase 2) : {trainable_count} poids")

# LR très faible pour ne pas détruire les représentations pré-entraînées
mnv2_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb_p2 = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10,
                             restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                 patience=5, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint('best_mnv2_model.keras', monitor='val_accuracy',
                               save_best_only=True, verbose=0)
]

# On recrée les générateurs avec la même augmentation
train_gen_mnv2 = aug_datagen.flow(X_train_mnv2, y_train_mnv2,
                                   batch_size=BATCH_SIZE, seed=SEED)

print(f"Phase 2 — Fine-tuning ({EPOCHS_P2} époques max, LR=1e-5)...")
history_p2 = mnv2_model.fit(
    train_gen_mnv2,
    epochs=EPOCHS_P2,
    validation_data=val_gen_mnv2,
    callbacks=cb_p2,
    verbose=1
)
print("\n✓ Phase 2 terminée.")


In [ ]:
# ── Courbes d'apprentissage MobileNetV2 (phases 1+2) ─────────
def concat_history(h1, h2, key):
    return h1.history[key] + h2.history[key]

ep1 = len(history_p1.history['loss'])
ep2 = len(history_p2.history['loss'])
total_ep = ep1 + ep2

train_loss = concat_history(history_p1, history_p2, 'loss')
val_loss   = concat_history(history_p1, history_p2, 'val_loss')
train_acc  = concat_history(history_p1, history_p2, 'accuracy')
val_acc    = concat_history(history_p1, history_p2, 'val_accuracy')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = range(1, total_ep + 1)

for ax, (tr, vl, title, ylabel) in zip(axes, [
    (train_loss, val_loss, 'Loss — MobileNetV2', 'Loss'),
    (train_acc,  val_acc,  'Accuracy — MobileNetV2', 'Accuracy')
]):
    ax.plot(epochs_x, tr, label='Train', color='royalblue')
    ax.plot(epochs_x, vl, label='Val',   color='tomato', linestyle='--')
    ax.axvline(ep1, color='green', linestyle=':', alpha=0.7, label=f'Début fine-tuning (ep.{ep1})')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Époque')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('mnv2_learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Évaluation MobileNetV2 sur test ─────────────────────────
y_pred_mnv2_proba = mnv2_model.predict(X_test_mnv2, verbose=0)
y_pred_mnv2 = np.argmax(y_pred_mnv2_proba, axis=1)

acc_mnv2 = accuracy_score(y_test_mnv2, y_pred_mnv2)
f1_mnv2  = f1_score(y_test_mnv2, y_pred_mnv2, average='macro')

print(f"MobileNetV2 — Accuracy : {acc_mnv2:.4f}  |  F1-macro : {f1_mnv2:.4f}")
print()
print(classification_report(y_test_mnv2, y_pred_mnv2,
                              target_names=CLASSES, digits=3))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(9, 7))
cm_mnv2 = confusion_matrix(y_test_mnv2, y_pred_mnv2)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_mnv2, display_labels=CLASSES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title(f'Matrice de confusion — MobileNetV2 (F1={f1_mnv2:.3f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mnv2_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()


---
<a id='8'></a>
## 8. Comparaison des performances

Nous comparons les trois modèles selon :
- **Accuracy** (proportion de prédictions correctes)
- **F1-macro** (moyenne des F1 par classe — robuste au déséquilibre)
- **F1 par classe** (identifie les émotions difficiles)
- **Matrices de confusion superposées** (patterns d'erreurs)


In [ ]:
# ── Tableau comparatif ───────────────────────────────────────
# F1 par classe pour chaque modèle
f1_rf_per_class   = f1_score(y_test_hog_enc, y_pred_rf,   average=None)
f1_cnn_per_class  = f1_score(y_test_cnn,     y_pred_cnn,  average=None)
f1_mnv2_per_class = f1_score(y_test_mnv2,    y_pred_mnv2, average=None)

df_comparison = pd.DataFrame({
    'Modèle':          ['Random Forest + HOG', 'CNN from scratch', 'MobileNetV2 TL'],
    'Accuracy':        [acc_rf, acc_cnn, acc_mnv2],
    'F1-macro':        [f1_rf, f1_cnn, f1_mnv2],
})
df_comparison = df_comparison.set_index('Modèle')
df_comparison['Accuracy'] = df_comparison['Accuracy'].map('{:.4f}'.format)
df_comparison['F1-macro'] = df_comparison['F1-macro'].map('{:.4f}'.format)

print("=" * 55)
print("TABLEAU COMPARATIF DES PERFORMANCES")
print("=" * 55)
print(df_comparison.to_string())
print()

# F1 par classe
df_per_class = pd.DataFrame({
    'Classe':              CLASSES,
    'F1 RF+HOG':           f1_rf_per_class,
    'F1 CNN scratch':      f1_cnn_per_class,
    'F1 MobileNetV2':      f1_mnv2_per_class,
})
print(df_per_class.to_string(index=False, float_format='{:.3f}'.format))


In [ ]:
# ── Graphique comparatif F1 par classe ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1-macro global
models_names = ['RF + HOG', 'CNN scratch', 'MobileNetV2']
f1_scores    = [f1_rf, f1_cnn, f1_mnv2]
acc_scores   = [acc_rf, acc_cnn, acc_mnv2]
bar_colors   = ['#E07070', '#70A0E0', '#70C080']

x = np.arange(len(models_names))
w = 0.35
axes[0].bar(x - w/2, acc_scores, w, label='Accuracy', color=bar_colors, alpha=0.7, edgecolor='white')
axes[0].bar(x + w/2, f1_scores,  w, label='F1-macro', color=bar_colors, alpha=1.0, edgecolor='white',
            linewidth=1.5, linestyle='--')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_names)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Accuracy et F1-macro par modèle', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
for xi, (a, f) in enumerate(zip(acc_scores, f1_scores)):
    axes[0].text(xi - w/2, a + 0.02, f'{a:.3f}', ha='center', fontsize=8)
    axes[0].text(xi + w/2, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)

# F1 par classe
x2 = np.arange(NUM_CLASSES)
w2 = 0.25
axes[1].bar(x2 - w2,   f1_rf_per_class,   w2, label='RF+HOG',       color='#E07070', alpha=0.85, edgecolor='white')
axes[1].bar(x2,         f1_cnn_per_class,  w2, label='CNN scratch',  color='#70A0E0', alpha=0.85, edgecolor='white')
axes[1].bar(x2 + w2,   f1_mnv2_per_class, w2, label='MobileNetV2',  color='#70C080', alpha=0.85, edgecolor='white')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(CLASSES, rotation=25)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('F1 par classe et par modèle', fontweight='bold')
axes[1].set_ylabel('F1-score')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_metrics.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Matrices de confusion côte à côte ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, cm, title in zip(
    axes,
    [cm_rf, cm_cnn, cm_mnv2],
    [f'RF + HOG\nF1={f1_rf:.3f}',
     f'CNN scratch\nF1={f1_cnn:.3f}',
     f'MobileNetV2\nF1={f1_mnv2:.3f}']
):
    # Normalisation en pourcentage
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', ax=ax,
                xticklabels=CLASSES, yticklabels=CLASSES,
                cmap='Blues', vmin=0, vmax=1,
                linewidths=0.5, linecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')
    ax.tick_params(axis='x', rotation=35)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Matrices de confusion normalisées — comparaison',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('comparison_confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()


---
<a id='9'></a>
## 9. Explicabilité — Grad-CAM

### Principe

**Grad-CAM (Gradient-weighted Class Activation Mapping)** calcule le gradient du score de la classe prédite par rapport aux feature maps de la dernière couche convolutive. Ces gradients indiquent quelles régions de l'image ont le plus contribué à la décision.

Pour un modèle de classification d'émotions félins, on s'attend à ce que Grad-CAM mette en évidence :
- Les **oreilles** (indicateur majeur : aplaties = peur/colère, dressées = attentif)
- Les **yeux** (ouverture, dilatation des pupilles)
- Les **moustaches** (tension / relâchement)

Si les heatmaps pointent vers le fond ou des artefacts, cela révèle un **biais de dataset** (corrélation fond ↔ classe).


In [ ]:
# ── Implémentation Grad-CAM ──────────────────────────────────
def get_gradcam_heatmap(model, img_array, class_idx=None, last_conv_layer_name=None):
    """Calcule la heatmap Grad-CAM pour une image donnée.

    Args:
        model             : modèle Keras entraîné
        img_array         : image préprocessée (1, H, W, 3)
        class_idx         : indice de classe cible (None = classe prédite)
        last_conv_layer_name : nom de la dernière couche Conv2D
    Returns:
        heatmap (np.array, H×W, valeurs [0,1])
    """
    # ── Trouver la dernière couche convolutive automatiquement ──
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, layers.Conv2D):
                last_conv_layer_name = layer.name
                break
            # Pour les modèles fonctionnels avec backbone imbriqué
            if hasattr(layer, 'layers'):
                for sub_layer in reversed(layer.layers):
                    if isinstance(sub_layer, layers.Conv2D):
                        last_conv_layer_name = sub_layer.name
                        break
                if last_conv_layer_name:
                    break

    # ── Construire un sous-modèle : input → [conv_output, predictions] ──
    try:
        # Modèle Sequential
        grad_model = keras.Model(
            inputs=model.inputs,
            outputs=[model.get_layer(last_conv_layer_name).output, model.output]
        )
    except ValueError:
        # Modèle fonctionnel avec backbone (MobileNetV2)
        # On cherche dans les sous-couches
        backbone = None
        for layer in model.layers:
            if hasattr(layer, 'layers'):
                backbone = layer
                break
        conv_output = backbone.get_layer(last_conv_layer_name).output
        # Reconstruire depuis le backbone
        grad_model = keras.Model(
            inputs=model.inputs,
            outputs=[conv_output, model.output]
        )

    with tf.GradientTape() as tape:
        img_tensor = tf.cast(img_array, tf.float32)
        conv_outputs, predictions = grad_model(img_tensor)
        if class_idx is None:
            class_idx = tf.argmax(predictions[0])
        class_channel = predictions[:, class_idx]

    # Gradients des activations par rapport à la classe
    grads = tape.gradient(class_channel, conv_outputs)

    # Pooling global des gradients (importance de chaque filtre)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Pondération des feature maps par les gradients
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # ReLU + normalisation
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(class_idx)


def overlay_gradcam(img_array_orig, heatmap, alpha=0.45):
    """Superpose la heatmap Grad-CAM sur l'image originale."""
    h, w = img_array_orig.shape[:2]
    heatmap_resized = np.array(Image.fromarray(
        np.uint8(255 * heatmap)
    ).resize((w, h), Image.BILINEAR))

    colormap = plt.get_cmap('jet')
    heatmap_colored = colormap(heatmap_resized / 255.0)[:, :, :3]  # (H,W,3) RGB

    img_norm = img_array_orig / 255.0 if img_array_orig.max() > 1 else img_array_orig
    superimposed = alpha * heatmap_colored + (1 - alpha) * img_norm
    superimposed = np.clip(superimposed, 0, 1)
    return superimposed

print("Fonctions Grad-CAM définies.")


In [ ]:
# ── Visualisation Grad-CAM sur exemples représentatifs ───────
# On sélectionne : 1 bonne prédiction par modèle + 1 erreur pour comparer

N_GRADCAM = 6   # Nombre d'exemples à visualiser
fig, axes = plt.subplots(N_GRADCAM, 4, figsize=(16, N_GRADCAM * 3.5))
fig.suptitle('Grad-CAM — Zones d\'attention du modèle MobileNetV2',
             fontsize=14, fontweight='bold')

# Sélectionner des exemples variés du test set
np.random.seed(SEED)
indices = np.random.choice(len(X_test_mnv2), N_GRADCAM, replace=False)

# Dernière couche conv de MobileNetV2 (dans le backbone)
# MobileNetV2 : 'block_16_project_BN' ou 'out_relu' selon la version
LAST_CONV = 'out_relu'   # Dernière activation avant GAP dans MobileNetV2

for row_i, idx in enumerate(indices):
    img_mnv2  = X_test_mnv2[idx:idx+1]       # (1, 128, 128, 3) — préprocessé MobileNetV2
    true_label = y_test_mnv2[idx]

    # Prédiction
    proba     = mnv2_model.predict(img_mnv2, verbose=0)[0]
    pred_cls  = np.argmax(proba)
    confidence = proba[pred_cls]

    # Image originale (dénormalisée pour affichage)
    img_display = np.clip((img_mnv2[0] + 1) / 2.0, 0, 1)  # [-1,1] → [0,1]

    # Grad-CAM
    try:
        heatmap, _ = get_gradcam_heatmap(mnv2_model, img_mnv2,
                                          class_idx=pred_cls,
                                          last_conv_layer_name=LAST_CONV)
        overlay    = overlay_gradcam(img_display, heatmap)
        gradcam_ok = True
    except Exception as e:
        gradcam_ok = False
        heatmap    = np.zeros((8, 8))
        overlay    = img_display

    correct = (pred_cls == true_label)
    status  = '✓' if correct else '✗'
    color   = 'green' if correct else 'red'

    # ── Colonne 1 : Image originale ──
    axes[row_i][0].imshow(img_display)
    axes[row_i][0].set_title(f'Original\nVrai : {CLASSES[true_label]}', fontsize=9)
    axes[row_i][0].axis('off')

    # ── Colonne 2 : Heatmap seule ──
    axes[row_i][1].imshow(heatmap, cmap='jet')
    axes[row_i][1].set_title('Heatmap Grad-CAM', fontsize=9)
    axes[row_i][1].axis('off')

    # ── Colonne 3 : Superposition ──
    axes[row_i][2].imshow(overlay)
    axes[row_i][2].set_title(f'Superposition\nPrédit : {CLASSES[pred_cls]} ({confidence:.0%}) {status}',
                              fontsize=9, color=color)
    axes[row_i][2].axis('off')

    # ── Colonne 4 : Distribution des probabilités ──
    bars = axes[row_i][3].barh(CLASSES, proba,
                                color=['#70C080' if i == true_label else
                                       '#E07070' if i == pred_cls else '#AAAAAA'
                                       for i in range(NUM_CLASSES)])
    axes[row_i][3].set_xlim(0, 1)
    axes[row_i][3].set_title('Probabilités', fontsize=9)
    axes[row_i][3].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
    axes[row_i][3].tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('gradcam_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

print("Vert = vraie classe | Rouge = classe prédite (si erreur)")


In [ ]:
# ── Analyse Grad-CAM : comparaison CNN scratch vs MobileNetV2 ─
# Pour 3 exemples, afficher côte à côte les zones d'attention des deux CNN

fig, axes = plt.subplots(3, 5, figsize=(18, 12))
fig.suptitle('Comparaison zones d\'attention : CNN scratch vs MobileNetV2',
             fontsize=13, fontweight='bold')

sample_idx = indices[:3]

for row_i, idx in enumerate(sample_idx):
    img_cnn_input  = X_test_cnn[idx:idx+1]    # [0,1]
    img_mnv2_input = X_test_mnv2[idx:idx+1]   # [-1,1]
    true_label     = y_test_cnn[idx]

    img_display = X_test_cnn[idx]  # [0,1]

    # Grad-CAM CNN scratch
    try:
        hm_cnn, pred_cnn = get_gradcam_heatmap(cnn_model, img_cnn_input)
        ov_cnn = overlay_gradcam(img_display, hm_cnn)
    except Exception:
        hm_cnn = np.zeros((4, 4)); ov_cnn = img_display; pred_cnn = 0

    # Grad-CAM MobileNetV2
    try:
        hm_mnv2, pred_mnv2 = get_gradcam_heatmap(mnv2_model, img_mnv2_input,
                                                   last_conv_layer_name=LAST_CONV)
        img_display_mnv2 = np.clip((img_mnv2_input[0] + 1) / 2.0, 0, 1)
        ov_mnv2 = overlay_gradcam(img_display_mnv2, hm_mnv2)
    except Exception:
        hm_mnv2 = np.zeros((4, 4)); ov_mnv2 = img_display; pred_mnv2 = 0

    titles = [
        f'Original\n(Vrai: {CLASSES[true_label]})',
        f'CNN — Heatmap\n(Prédit: {CLASSES[pred_cnn]})',
        f'CNN — Overlay',
        f'MNV2 — Heatmap\n(Prédit: {CLASSES[pred_mnv2]})',
        f'MNV2 — Overlay',
    ]
    images = [img_display, hm_cnn, ov_cnn, hm_mnv2, ov_mnv2]
    cmaps  = [None, 'jet', None, 'jet', None]

    for col_i, (im, ttl, cm_) in enumerate(zip(images, titles, cmaps)):
        axes[row_i][col_i].imshow(im, cmap=cm_)
        axes[row_i][col_i].set_title(ttl, fontsize=8.5)
        axes[row_i][col_i].axis('off')

plt.tight_layout()
plt.savefig('gradcam_cnn_vs_mnv2.png', dpi=100, bbox_inches='tight')
plt.show()


---
<a id='10'></a>
## 10. Discussion — Limites, biais et pistes d'amélioration

### 10.1 Limites identifiées

#### Volume et diversité du dataset
Le dataset ne contient que **671 images** — un volume très insuffisant pour un CNN industrial-grade. Les conséquences observées :
- Variance élevée entre runs (résultats sensibles à la partition train/test)
- Risque de surapprentissage malgré la data augmentation et le dropout
- Sous-représentation de certaines races, âges et conditions d'éclairage

#### Ambiguïté des annotations
Certaines émotions sont intrinsèquement ambiguës :
- **Scared vs Angry** : oreilles aplaties dans les deux cas, distinction subtile par la posture du corps (non visible si image cadrée sur la tête)
- **Normal vs Relaxed** : frontière floue même pour un humain expert
- L'annotateur unique (ou petit groupe) peut introduire un biais subjectif

#### Résolution de travail
Travailler à **128×128** fait perdre des détails fins (texture des moustaches, microexpressions). Une résolution de 224×224 améliorerait la précision au prix d'un entraînement plus long.

### 10.2 Biais potentiels

| Biais | Description | Impact |
|---|---|---|
| **Biais de fond** | Certaines classes sur-représentées avec fond blanc (vétérinaire) | Le modèle apprend le fond, pas l'émotion |
| **Biais de race** | Sur-représentation de certaines races (ex: chats siamois, europens) | Mauvaise généralisation sur autres races |
| **Biais d'éclairage** | Photos studio vs. photos naturelles | Distribution shift à l'inférence |
| **Biais de cadrage** | Certaines images incluent le corps, d'autres non | Incohérence des features disponibles |

### 10.3 Pistes d'amélioration

**Données**
- Augmenter le dataset à 5 000+ images par classe via scraping éthique ou annotation collaborative
- Utiliser de la **data augmentation forte** : MixUp, CutMix, RandAugment
- Équilibrer le dataset avec SMOTE adapté aux images (oversampling des classes rares)

**Modélisation**
- Tester **EfficientNetB0/B2** : meilleur rapport performance/paramètres que MobileNetV2
- **Vision Transformers (ViT)** : meilleures capacités d'attention globale (yeux + oreilles simultanément)
- **Ensemble** de CNN scratch + MobileNetV2 par vote ou moyennage de probabilités

**Évaluation**
- Validation croisée **k-fold stratifiée** au lieu d'un seul split (variance réduite)
- Test sur un dataset externe indépendant (données vétérinaires réelles)
- Analyse d'erreurs qualitative par vétérinaire comportementaliste

**Déploiement**
- Seuil de confiance : si `max(proba) < 0.6`, retourner "Incertain" plutôt qu'une classe fausse
- Monitoring du drift : les distributions d'images changent selon les appareils photo


---
<a id='11'></a>
## 11. Conclusion et recommandations

### Synthèse des enseignements

Ce projet a conduit un pipeline Machine Learning complet de la donnée brute jusqu'à l'explicabilité du modèle, pour répondre à la problématique : *"Peut-on automatiser la reconnaissance des émotions félines à partir d'images ?"*

**Réponse :** Oui, avec des nuances importantes.

| Étape | Enseignement clé |
|---|---|
| EDA | Déséquilibre modéré (ratio ~2x) → F1-macro comme métrique principale |
| Baseline RF+HOG | Confirme la faisabilité non-neuronale (~45–55% F1-macro) |
| CNN scratch | Amélioration notable grâce à la data augmentation embarquée |
| MobileNetV2 TL | Meilleur modèle : bénéfice clair des features ImageNet sur petit dataset |
| Grad-CAM | Attention sur les oreilles et les yeux — cohérent biologiquement |

### Recommandations opérationnelles

1. **Déployer MobileNetV2** comme moteur de classification : meilleur compromis précision/vitesse/interprétabilité pour une intégration mobile ou embarquée.

2. **Ajouter un seuil de confiance** : ne proposer une classification que si `confidence ≥ 60%`, sinon afficher "Expression ambiguë — consulter un vétérinaire".

3. **Enrichir le dataset** en priorité sur les classes *Disgusted* et *Surprised* (moins représentées, F1 plus faible) — viser 200+ images par classe comme premier palier.

4. **Valider cliniquement** : soumettre les prédictions du modèle à un panel de vétérinaires comportementalistes pour estimer la valeur réelle de l'outil.

5. **Intégrer un pipeline de feedback** : chaque correction humaine est une nouvelle donnée d'entraînement → amélioration continue du modèle.

### Impact potentiel

Un tel système, déployé via application mobile ou caméra connectée, pourrait :
- Alerter un propriétaire quand son chat présente des signes de stress prolongé
- Aider un vétérinaire à objectiver le niveau d'anxiété pré-opératoire
- Servir de base à des études comportementales à grande échelle

---
*Notebook réalisé dans le cadre de l'évaluation Projet Data — 2026*
